# Глава 25. Этика ИИ: справедливость и предвзятость

_Ноутбук сгенерирован из учебника «Машинное обучение для начинающих». Запускайте ячейки по порядку сверху вниз в Google Colab._

**Цель главы  🎯**

После этой главы ты сможешь объяснить, откуда в моделях берётся предвзятость (bias), проверить модель на справедливость между группами и применить чек-лист этики перед защитой проекта.

Модель делает то, чему её научили данные. Если в данных отражена человеческая несправедливость — модель её впитает и будет воспроизводить, причём в масштабе и «объективным» на вид способом. Это не гипотетическая проблема будущего, а то, что уже происходит и с чем обязан считаться каждый, кто строит модели.

**Аналогия: зеркало общества  💡**

Модель — как зеркало, отражающее данные, на которых она училась. Если в прошлых решениях людей была предвзятость (например, кому давали кредит или работу), модель отразит и усилит её, выдавая несправедливость за «математически обоснованное» решение. Зеркало не злое — оно просто честно отражает то, что ему показали. Наша задача — заметить искажение и исправить его.

## Откуда берётся предвзятость (bias)

Предвзятость в ML (bias) — это систематическая несправедливость модели по отношению к какой-то группе людей. Она возникает в основном из данных, а не из «злого умысла» алгоритма. Главные источники: перекос в данных (одной группы в обучении намного больше), историческая несправедливость (данные отражают прошлые предвзятые решения) и плохие признаки (модель использует пол, расу или их «заменители» как основу решения).

## Три реальных случая, которые изменили индустрию

Это не выдуманные примеры — реальные системы, чьи ошибки стали уроками для всей отрасли.

Система

Что пошло не так

COMPAS (суды США)

Система оценки риска рецидива чаще ошибочно записывала темнокожих подсудимых в «высокий риск»

Amazon (наём)

ИИ-отбор резюме занижал оценки женщинам — учился на прошлых наймах, где преобладали мужчины

Распознавание лиц

Точность заметно ниже для женщин и людей с тёмной кожей — в обучении их было мало

Общий урок всех трёх случаев: модель не была «злой», она честно выучила перекосы своих данных. Никто не заметил проблему вовремя, потому что не проверял модель на справедливость между группами. Именно этому мы сейчас научимся.

## Ключевые понятия

Термин

Простыми словами

Protected attributes

Защищённые признаки: пол, раса, возраст, религия — по ним нельзя дискриминировать

Fairness (справедливость)

Модель одинаково хорошо работает для разных групп людей

Disparate impact

Непропорциональный вред: модель формально нейтральна, но бьёт по одной группе сильнее

Proxy (прокси-признак)

Признак-заменитель: район проживания может «выдавать» расу, имя — пол

**Внимание: убрать признак — не значит убрать предвзятость  ⚠️**

Наивно думать, что достаточно удалить пол или расу из данных. Модель может восстановить их через прокси-признаки: имя, район, даже хобби коррелируют с полом и происхождением. Поэтому предвзятость проверяют не по наличию признака, а по результату — одинаково ли справедлива модель к разным группам.

## Практика: проверь пингвинов на справедливость

Проверим, одинаково ли хорошо наша модель предсказывает для пингвинов с разных островов. Разная точность между группами — сигнал предвзятости.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

url = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/penguins.csv"
df = pd.read_csv(url).dropna(subset=["bill_length_mm", "bill_depth_mm", "body_mass_g", "species"])

X = df[["bill_length_mm", "bill_depth_mm", "body_mass_g"]]
y = df["species"]
island = df["island"]   # признак группы — проверяем справедливость по нему

X_tr, X_te, y_tr, y_te, island_tr, island_te = train_test_split(
    X, y, island, test_size=0.3, random_state=42)

model = RandomForestClassifier(random_state=42).fit(X_tr, y_tr)
pred = model.predict(X_te)

# Считаем точность ОТДЕЛЬНО для каждой группы
for group in ["Biscoe", "Dream", "Torgersen"]:
    mask = (island_te == group)
    acc = accuracy_score(y_te[mask], pred[mask])
    print(f"Точность для {group}: {acc:.3f}")

_Если точность для групп сильно различается — это сигнал возможной предвзятости._

Обрати внимание: мы не подавали остров в модель как признак, но всё равно проверяем результат по островам. Именно так ищут скрытую несправедливость — сравнивая качество модели между группами, а не доверяя тому, что «такого признака в модели нет».

## Локальный пример: кредитный скоринг

Представь, что банк в Душанбе строит модель одобрения кредита. Если исторически кредиты чаще одобряли жителям столицы, модель выучит «регион → одобрение» и будет отказывать жителям районов, даже если убрать регион из признаков (его выдаст, например, почтовый индекс — прокси-признак). Проверка справедливости между регионами здесь так же обязательна, как между островами в нашем примере с пингвинами.

## Чек-лист «Проверь модель на предвзятость»

**Перед защитой финального проекта  🎯**

Есть ли в данных защищённые признаки (пол, возраст, регион, национальность)?

Сбалансированы ли группы в данных или одной сильно больше?

Отличается ли точность (или recall) модели между группами?

Нет ли прокси-признаков, которые «выдают» защищённый признак?

Понимаю ли я, кому навредит ошибка модели и насколько это серьёзно?

Могу ли я объяснить решение модели человеку, которого оно касается?

## Задания

Базовый уровень

Запусти проверку на пингвинах: одинакова ли точность модели для разных островов? Затем сгруппируй по виду (species) и проверь так же. Есть ли группа, для которой модель работает заметно хуже?

Со звёздочкой  ⭐

Возьми любой свой датасет с признаком-группой (пол, возраст, регион). Проверь, одинаково ли работает модель для групп. Если нашёл перекос — предложи, как его уменьшить (больше данных группы, class_weight, другой признак). Опиши этическую дилемму, если она есть.

Вопросы для обсуждения  💬

Кто несёт ответственность за предвзятое решение ИИ — программист, компания или «сам алгоритм»?

Достаточно ли удалить признак «пол» из данных, чтобы модель стала справедливой? Почему нет?

Приведи пример из своей страны, где предвзятая модель могла бы навредить людям.

**Проверь себя  ℹ️**

1. Откуда в модели берётся предвзятость?

2. Что такое прокси-признак и почему он опасен?

3. Как проверить модель на справедливость между группами?